In [ ]:
import warnings
import pandas as pd
import optuna

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import f1_score, roc_auc_score

from xgboost import XGBClassifier


# 경고 숨김부
warnings.filterwarnings("ignore")

# 결과 출력 형식 설정부
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

# 파일 경로 설정부
train_valid_file_path = r"C:\myCode\ott-churn-prediction\kim.kwangil\data\03_1차 파생\260602_최종파생변수(80개).csv"
test_file_path = r"C:\myCode\ott-churn-prediction\kim.kwangil\data\03_1차 파생\260602_테스트용(80개).csv"

# 모델 입력 제외 컬럼 설정부
exclude_cols = [
    "USER_KEY",
    "product_code",
    "price",
    "billing_method",
    "payment_device",
    "gender",
    "age",
    "reg_date",
    "reg_hour",
    "end_date",
    "is_repurchase",
]


# 이진 변수 변환 함수부
def to_binary(series):
    lowered = series.astype(str).str.strip().str.lower()

    mapped = lowered.map(
        {
            "1": 1,
            "0": 0,
            "y": 1,
            "n": 0,
            "yes": 1,
            "no": 0,
            "true": 1,
            "false": 0,
        }
    )

    numeric = pd.to_numeric(series, errors="coerce")

    return mapped.where(mapped.notna(), numeric)


# OneHotEncoder 버전 호환 함수부
def make_onehot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )


# 타깃 분포 출력 함수부
def print_target_distribution(y):
    target_dist = (
        pd.DataFrame({"count": y.value_counts().sort_index()})
        .assign(percent=lambda x: (x["count"] / x["count"].sum() * 100).round(2))
    )

    print(target_dist)


# 전처리 파이프라인 생성 함수부
def make_preprocessor(numeric_features, categorical_features):
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_onehot_encoder()),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )

    return preprocessor


# 데이터 전처리 함수부
def prepare_data(df, selected_features, numeric_features):
    df = df.copy()

    for col in numeric_features:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["is_repurchase_num"] = to_binary(df["is_repurchase"])
    df = df[df["is_repurchase_num"].isin([0, 1])].copy()

    X = df[selected_features].copy()
    y = (df["is_repurchase_num"] == 0).astype(int)

    return X, y


# 데이터 로드부
train_valid_df = pd.read_csv(train_valid_file_path).copy()
test_df = pd.read_csv(test_file_path).copy()

# 전체 컬럼 기준 입력 변수 생성부
selected_features = [
    col for col in train_valid_df.columns if col not in exclude_cols
]

# 범주형 변수 설정부
categorical_features = [
    "age_group",
]

# 숫자형 변수 설정부
numeric_features = [
    col for col in selected_features if col not in categorical_features
]

# 입력 변수, 타깃 변수 생성부
X_train_valid, y_train_valid = prepare_data(
    train_valid_df,
    selected_features,
    numeric_features,
)

X_test, y_test = prepare_data(
    test_df,
    selected_features,
    numeric_features,
)

print("분석 기준: 최종 파생 변수 + XGBoost Optuna 5-Fold 튜닝")
print("양성 클래스 기준: is_repurchase == 0")
print(f"train/valid 데이터 수: {len(X_train_valid)}")
print(f"test 데이터 수: {len(X_test)}")
print(f"전체 데이터 수: {len(X_train_valid) + len(X_test)}")
print(f"train/valid 비율: {len(X_train_valid) / (len(X_train_valid) + len(X_test)) * 100:.2f}%")
print(f"test 비율: {len(X_test) / (len(X_train_valid) + len(X_test)) * 100:.2f}%")
print(f"train/valid 파일 전체 컬럼 수: {len(train_valid_df.columns)}")
print(f"test 파일 전체 컬럼 수: {len(test_df.columns)}")
print(f"모델 입력 제외 컬럼 수: {len(exclude_cols)}")
print(f"사용 변수 수: {len(selected_features)}")
print("사용 변수")
print(selected_features)
print("train/valid 타깃 분포")
print_target_distribution(y_train_valid)
print("test 타깃 분포")
print_target_distribution(y_test)

if y_train_valid.nunique() < 2 or y_train_valid.value_counts().min() < 5:
    print("학습 불가: train/valid 타깃 클래스가 부족합니다.")
elif y_test.nunique() < 2:
    print("평가 불가: test 타깃 클래스가 부족합니다.")
else:
    # XGBoost 불균형 가중치 계산부
    negative_count = (y_train_valid == 0).sum()
    positive_count = (y_train_valid == 1).sum()
    scale_pos_weight = negative_count / positive_count if positive_count > 0 else 1

    # Optuna 목적 함수부
    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "max_depth": trial.suggest_int("max_depth", 2, 10),
            "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 20.0, log=True),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "gamma": trial.suggest_float("gamma", 0.0, 10.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
            "scale_pos_weight": scale_pos_weight,
            "objective": "binary:logistic",
            "eval_metric": "logloss",
            "random_state": 42,
            "n_jobs": -1,
        }

        skf = StratifiedKFold(
            n_splits=5,
            shuffle=True,
            random_state=42,
        )

        fold_scores = []

        for train_idx, valid_idx in skf.split(X_train_valid, y_train_valid):
            X_fold_train = X_train_valid.iloc[train_idx]
            X_fold_valid = X_train_valid.iloc[valid_idx]
            y_fold_train = y_train_valid.iloc[train_idx]
            y_fold_valid = y_train_valid.iloc[valid_idx]

            clf = Pipeline(
                steps=[
                    ("preprocessor", make_preprocessor(numeric_features, categorical_features)),
                    ("model", XGBClassifier(**params)),
                ]
            )

            clf.fit(X_fold_train, y_fold_train)

            valid_proba = clf.predict_proba(X_fold_valid)[:, 1]
            valid_roc_auc = roc_auc_score(y_fold_valid, valid_proba)

            fold_scores.append(valid_roc_auc)

        return sum(fold_scores) / len(fold_scores)

    # Optuna 튜닝 수행부
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42),
    )

    study.optimize(
        objective,
        n_trials=200,
        show_progress_bar=True,
    )

    print("Optuna best 5-fold valid roc_auc")
    print(round(study.best_value, 4))
    print("Optuna best params")
    print(study.best_params)

    # 최적 파라미터 기반 최종 모델 학습부
    best_params = {
        **study.best_params,
        "scale_pos_weight": scale_pos_weight,
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "random_state": 42,
        "n_jobs": -1,
    }

    print("최종 학습에 사용한 XGBoost 파라미터")
    print(best_params)

    final_clf = Pipeline(
        steps=[
            ("preprocessor", make_preprocessor(numeric_features, categorical_features)),
            ("model", XGBClassifier(**best_params)),
        ]
    )

    final_clf.fit(X_train_valid, y_train_valid)

    y_train_valid_pred = final_clf.predict(X_train_valid)
    y_test_pred = final_clf.predict(X_test)

    y_train_valid_proba = final_clf.predict_proba(X_train_valid)[:, 1]
    y_test_proba = final_clf.predict_proba(X_test)[:, 1]

    train_valid_roc_auc = roc_auc_score(y_train_valid, y_train_valid_proba)
    test_roc_auc = roc_auc_score(y_test, y_test_proba)
    auc_gap = train_valid_roc_auc - test_roc_auc

    result = {
        "model": "XGBoost_Optuna_5Fold",
        "f1_score": f1_score(y_test, y_test_pred, zero_division=0),
        "train_valid_roc_auc": train_valid_roc_auc,
        "test_roc_auc": test_roc_auc,
        "auc_gap": auc_gap,
        "is_overfit": auc_gap >= 0.05,
    }

    results_df = (
        pd.DataFrame([result])
        .set_index("model")
        [
            [
                "f1_score",
                "train_valid_roc_auc",
                "test_roc_auc",
                "auc_gap",
                "is_overfit",
            ]
        ]
        .round(4)
    )

    print("최종 테스트 성능")
    print(results_df.to_string())